# Домашнее задание 1
## Датасет:
* Online shoppers Intention
## Гипотезы:
* Random Forest покажет более высокий F1 для Revenue, чем KNN и Gaussian Naive Bayes, из-за способности учитывать нелинейные взаимодействия признаков
* Увеличение числа соседей с 1 до 15 для KNN повысит F1, поскольку уменьшится влияние отдельных нетипичных посещений(выбросов)
* Удаление ProductRelated сильнее снизит F1, чем удаление времени на них ProductRelated_Duration, так как колво просмотренных страниц важнее
* Для RandomForest среди глубин 3–15 лучший средний F1 будет при 15, если остальные настройки одинаковые
* Самыми важными признаками в RF будут: VisitorType -> Weekend -> ProductRelated_Duration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             accuracy_score, roc_auc_score)
from sklearn.metrics import make_scorer
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Шаг 1 EDA

In [ ]:


SEED = 42

df = pd.read_csv('./data/online_shoppers_intention.csv')

df.info()
print(df.shape)
print(df.describe())
print(df.dtypes)
positive_amount = len(df[df["Revenue"] == True])
print("Positive revenue:", positive_amount)
print(f"True Revenue in dataset: {(positive_amount / len(df) * 100):.2f}%")
print(f"Average amount spent on items pages : {df["ProductRelated_Duration"].mean():.2f}")

print(f"Average amount spent on items pages on positive revenue: {df[df["Revenue"] == True]["ProductRelated_Duration"].mean():.2f}")
FEATURES = [str(i) for i in list(df.columns)]
print(FEATURES)
non_class_features = FEATURES[:10]
print(non_class_features)
feature_descriptions = {
    "Administrative": "Количество административных страниц",
    "Administrative_Duration": "Время на административных страницах, с",
    "Informational": "Количество информационных страниц",
    "Informational_Duration": "Время на информационных страницах, с",
    "ProductRelated": "Количество товарных страниц",
    "ProductRelated_Duration": "Время на товарных страницах, с",
    "BounceRates": "Показатель отказов посещённых страниц",
    "ExitRates": "Показатель выходов с посещённых страниц",
    "PageValues": "Денежная ценность посещённых страниц по веб-аналитике",
    "SpecialDay": "Близость визита к особому торговому дню",
    "Month": "Месяц посещения",
    "OperatingSystems": "Код операционной системы",
    "Browser": "Код браузера",
    "Region": "Код региона",
    "TrafficType": "Код источника трафика",
    "VisitorType": "Тип посетителя",
    "Weekend": "Посещение в выходной день",
    "Revenue": "Факт совершения покупки"
}

### Рассмотри как признаки связаны с тем купит человек что то или нет?

In [ ]:
import textwrap

ht = df.groupby("Revenue")[non_class_features].mean()
st = df.groupby("Revenue")[non_class_features].std()
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

x = np.arange(2)
width = 0.25


for ax, feature in zip(axes.flatten(), non_class_features):
    for i, spec in enumerate([True, False]):
        ax.bar(
            x[i],
            ht.loc[spec, feature],
            width=width,
            yerr=st.loc[spec, feature],
            capsize=4,
            color="orange" if spec else "steelblue"
        )
    ax.set_xticks(x)
    ax.set_xticklabels(["С покупкой", "Без покупки"])
    ax.set_title(
        textwrap.fill(feature_descriptions[feature], width=30)
    )
    ax.set_ylabel("Среднее значение")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()


### Splitting into train and test data -> Encoding data and normolizing it


In [ ]:
before = len(df)

df_no_duplicates = df.drop_duplicates().reset_index(drop=True)

print("Удалено повторов:", before - len(df_no_duplicates))
print("Осталось строк:", len(df_no_duplicates))

Y_full = df_no_duplicates["Revenue"]
X_full = df_no_duplicates.drop(columns=["Revenue"])
print(Y_full.shape)
print(X_full.shape)

X_train, X_test, y_train, y_test = train_test_split(X_full, Y_full, test_size=0.2, random_state=SEED, stratify=Y_full)

feature_to_be_encoded = ["Month",
    "VisitorType",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType"]

preprocesor = ColumnTransformer(
    transformers=[
        ("categories",
         OneHotEncoder(
             sparse_output=False,
             handle_unknown="ignore",
         ),
         feature_to_be_encoded),
        ("scaler", StandardScaler(), non_class_features),
    ],
    remainder="passthrough"
)

x_train = preprocesor.fit_transform(X_train)
x_test = preprocesor.transform(X_test)

print(f"x_train shape after encoding: {x_train.shape}")
print(f"x_test shape after encoding: {x_test.shape}")



# Делаем модели
## 1. KNN

In [ ]:

knn_pipeline = Pipeline([
    ("preprocessor", preprocesor),
    ("classifier", KNeighborsClassifier(
        metric="euclidean",
        weights="uniform",
        algorithm="brute"
    ))
])
param_grid = {
    "classifier__n_neighbors": list(range(1, 16))
}
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)
f1_scorer = make_scorer(
    f1_score,
    average="binary",
    pos_label=True,
    zero_division=0
)
knn_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=cv,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)
knn_search.fit(X_train, y_train)
results = pd.DataFrame(knn_search.cv_results_)
print(results[[
    "param_classifier__n_neighbors",
    "mean_test_score",
    "std_test_score"
]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(knn_search.cv_results_["param_classifier__n_neighbors"], knn_search.cv_results_["mean_test_score"], marker="o", label="model accuracy")
ax.legend()
ax.set_xlabel("k")
ax.set_ylabel("F1")
ax.set_title("F1 vs. k")
ax.set_xticks(knn_search.cv_results_["param_classifier__n_neighbors"])
plt.show()

print("\nЛучшие параметры:", knn_search.best_params_)
print(f"Лучший средний F1: {knn_search.best_score_:.4f}")
knn_model = knn_search.best_estimator_

## Баейс

In [ ]:

bayes_pipline = Pipeline(
    [
        ("preprocessor", preprocesor),
        ("classifier", GaussianNB()),
    ]
)
bayes_grid = {
    "classifier__var_smoothing": [float(f"1e-{i}") for i in range(0, 16)]
}

bayes_search = GridSearchCV(
    cv=cv,
    param_grid=bayes_grid,
    scoring=f1_scorer,
    estimator=bayes_pipline,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

bayes_search.fit(X_train, y_train)
results = pd.DataFrame(bayes_search.cv_results_)
print(results[[
    "param_classifier__var_smoothing",
    "mean_test_score",
    "std_test_score"
]].to_string(index=False))
print("\nЛучшие параметры:", bayes_search.best_params_)
print(f"Лучший средний F1: {bayes_search.best_score_:.4f}")
bayes_model = bayes_search.best_estimator_


## Random Forest


In [ ]:

RF_pipline = Pipeline(
    [
        ("preprocessor", preprocesor),
        ("classifier", RandomForestClassifier(max_depth=5, n_estimators=100, random_state=SEED)),
    ]
)
RF_grid = {
    "classifier__max_depth": list(range(3, 16)),
    "classifier__n_estimators": list(range(5, 100, 5)),
}

RF_search = GridSearchCV(
    cv=cv,
    param_grid=RF_grid,
    scoring=f1_scorer,
    estimator=RF_pipline,
    n_jobs=-1,
    refit=True,
    error_score="raise"
)

RF_search.fit(X_train, y_train)
results = pd.DataFrame(RF_search.cv_results_)
print(results[[
    "param_classifier__max_depth",
    "param_classifier__n_estimators",
    "mean_test_score",
    "std_test_score"
]].to_string(index=False))
print("\nЛучшие параметры:", RF_search.best_params_)
print(f"Лучший средний F1: {RF_search.best_score_:.4f}")
RF_model = RF_search.best_estimator_


## Тестируем все наши модели

In [ ]:
knn_accuracy = knn_model.score(X_test, y_test)
knn_f1 = f1_score(y_pred=knn_model.predict(X_test), y_true=y_test)
print(f"FINAL TEST KNN:\n"
      f"ACCURACY: {knn_accuracy:.4f}\n"
      f"F1 SCORE: {knn_f1:.4f}")

bayes_accuracy = bayes_model.score(X_test, y_test)
bayes_f1 = f1_score(y_pred=bayes_model.predict(X_test), y_true=y_test)
print(f"FINAL TEST BAYES:\n"
      f"ACCURACY: {bayes_accuracy:.4f}\n"
      f"F1 SCORE: {bayes_f1:.4f}")

RF_accuracy = RF_model.score(X_test, y_test)
RF_f1 = f1_score(y_pred=RF_model.predict(X_test), y_true=y_test)
print(f"FINAL TEST RF MODEL:\n"
      f"ACCURACY: {RF_accuracy:.4f}\n"
      f"F1 SCORE: {RF_f1:.4f}")


In [ ]:
model_names = ["KNN", "Gaussian Naive Bayes", "Random Forest"]
accuracy_values = [knn_accuracy, bayes_accuracy, RF_accuracy]
f1_values = [knn_f1, bayes_f1, RF_f1]
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(model_names))
width = 0.35
accuracy_bars = ax.bar(
    x - width / 2,
    accuracy_values,
    width=width,
    label="Accuracy",
    color="steelblue"
)
f1_bars = ax.bar(
    x + width / 2,
    f1_values,
    width=width,
    label="F1",
    color="orange"
)
ax.bar_label(accuracy_bars, fmt="%.3f", padding=3)
ax.bar_label(f1_bars, fmt="%.3f", padding=3)
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_xlabel("Модель")
ax.set_ylabel("Значение")
ax.set_title("Сравнение моделей на test")
ax.set_ylim(0, 1.1)
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

## Проверка оставшихся гипотез
#### Важность признаков RF

In [ ]:
rf_classifier = RF_model.named_steps["classifier"]
preproc = RF_model.named_steps["preprocessor"]
weight = rf_classifier.feature_importances_
feature_weight = []
for ind, i in enumerate(X_train.columns):
    feature_weight.append(0)
    for j, el in enumerate(preproc.get_feature_names_out()):
        if el.startswith("categories__" + i + "_") or (el == "scaler__" + i) or (el == "remainder__" + i):
            feature_weight[ind] += weight[j]

arr = np.argsort(feature_weight)[::-1]
print("\n".join([f"{el + 1}. {FEATURES[i]}" for el, i in enumerate(arr)]))
print(arr)

feat_importance = pd.Series(feature_weight, index=X_train.columns).sort_values(ascending=False)


fig, ax = (plt.subplots(figsize=(10, 7)))
feat_importance.sort_values().plot.barh(ax=ax)
ax.set_xlabel("важность")
ax.set_title("Важность признаков Random Forest")
ax.set_axisbelow(True)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

### Гипотеза Удаление признаков у Байеса

In [ ]:
from sklearn.base import clone

variants = {
    "All Features": [],
    "Without ProductRelated": ["ProductRelated"],
    "Without ProductRelated_Duration": ["ProductRelated_Duration"]
}
folds = list(cv.split(X_train, y_train))
comperer = []
for name, removed_columns in variants.items():
    x_var = X_train.drop(columns=removed_columns)
    changed_columns = [ column for column in non_class_features if column not in removed_columns]
    variant_preprocessor = ColumnTransformer(
        transformers=[
            (
                "categories",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                feature_to_be_encoded
            ),
            (
                "scaler",
                StandardScaler(),
                changed_columns
            )
        ],
        remainder="passthrough"
    )
    classifier = clone(bayes_model.named_steps["classifier"])
    model = Pipeline([
        ("preprocessor", variant_preprocessor),
        ("classifier", classifier)
    ])
    scores = cross_val_score(
        model,
        x_var,
        y_train,
        cv=folds,
        scoring=f1_scorer,
        n_jobs=-1,
        error_score="raise"
    )
    comperer.append({
        "Type": name,
        "F1": scores.mean(),
        "STD": scores.std()
    })

comperer = pd.DataFrame(comperer).set_index("Type")
baseline = comperer.loc["All Features", "F1"]
comperer["Снижение F1"] = baseline - comperer["F1"]
print(comperer.round(4))